# DuckDB + SLayer, from the command line

The same demo as the [Python notebook](duckdb_python_nb.ipynb), driven entirely through the `slayer` command line. We install the DuckDB CLI, expose a **48 KB CSV on a CDN** as a DuckDB view, let SLayer **auto-ingest** its schema into a semantic model, and query it with `slayer query`. Nothing is copied locally — the view points at the URL and every query reaches back over the wire.

Each `slayer query` runs in a shell cell; we ask it for `--format json` and let pandas render the rows as a table.

**Prerequisites:** `pip install motley-slayer`.

## 1. Set it up — all CLI

Three steps, no Python: install the DuckDB CLI, create a **view** over the remote CSV inside a DuckDB file, then point a SLayer datasource at that file and let `--ingest` **auto-build the model** by introspecting the view. Column names and types come straight from the data — nothing hand-written.

In [1]:
%%bash
set -euo pipefail
export SLAYER_STORAGE=.cache/cli/store
rm -rf .cache/cli && mkdir -p .cache/cli

# Install the DuckDB CLI if it isn't already on PATH (idempotent). The installer
# uses bash syntax, so pipe it to bash — /bin/sh is dash on many CI runners.
command -v duckdb >/dev/null 2>&1 || curl -fsSL https://install.duckdb.org | bash >/dev/null
export PATH="$HOME/.duckdb/cli/latest:$PATH"

# Expose the remote CSV as a DuckDB view: the CSV stays on the CDN, the view
# just points at it over httpfs — nothing is copied locally.
duckdb .cache/cli/weather.duckdb -c \
  "CREATE VIEW weather AS SELECT * FROM read_csv_auto('https://cdn.jsdelivr.net/npm/vega-datasets@2/data/seattle-weather.csv')"

# Register the datasource and auto-ingest the view into a semantic model —
# column names and types come from introspecting the view, nothing hand-written.
PYTHONWARNINGS=ignore slayer datasources create "duckdb:///$PWD/.cache/cli/weather.duckdb" \
  --name weather_db --ingest -y
slayer models list

Created datasource 'weather_db' (duckdb).
Ingested: weather (6 columns, 0 measures)


weather


## 2. A warm-up query

A plain grouped count — how many days of each weather type. The shell cell runs `slayer query ... --format json` and captures its output; the next line renders it with pandas.

In [2]:
%%bash --out warmup
export SLAYER_STORAGE=.cache/cli/store
slayer query '{"source_model": "weather", "dimensions": ["weather"], "measures": [{"formula": "*:count", "name": "days"}]}' --format json

In [3]:
import json

import pandas as pd
from IPython.display import Markdown

pd.DataFrame(json.loads(warmup))

,weather.weather,weather.days
0,drizzle,53
1,rain,641
2,sun,640
3,fog,101
4,snow,26


## 3. The hero query: an aggregate as a dimension, plus year-over-year

One query, two recent features. Stage 1 totals monthly rainfall and its value twelve months back (`time_shift(..., -1, 'year')`, calendar-aware). Stage 2 bands each month *rainy* or *dry* by that total inside a `CASE WHEN` **dimension** — an aggregate used as a dimension — and regroups. A SLayer query is itself a model, so stage 2 just reads stage 1 by name. The first year (2012) has nothing to look back to, so its year-over-year values are null.

We write the query to a file and run it with `slayer query @file` — the same JSON the Python notebook hands to the client.

In [4]:
%%bash --out hero_rows
set -e
export SLAYER_STORAGE=.cache/cli/store
cat > .cache/cli/hero.json <<'JSON'
[
  {
    "name": "monthly",
    "source_model": "weather",
    "time_dimensions": [{"dimension": "date", "granularity": "month"}],
    "measures": [
      {"formula": "precipitation:sum", "name": "rain"},
      {"formula": "precipitation:sum - time_shift(precipitation:sum, -1, 'year')", "name": "rain_yoy"}
    ]
  },
  {
    "source_model": "monthly",
    "dimensions": [
      {"expression": "CASE WHEN rain > 100 THEN 'rainy' ELSE 'dry' END", "name": "month_type"},
      "date"
    ],
    "measures": [
      {"formula": "rain:sum", "name": "total_rain"},
      {"formula": "rain_yoy:sum", "name": "total_rain_yoy"}
    ],
    "order": [{"column": "date", "direction": "asc"}]
  }
]
JSON
slayer query @.cache/cli/hero.json --format json

In [5]:
pd.DataFrame(json.loads(hero_rows))

,monthly.month_type,monthly.date,monthly.total_rain,monthly.total_rain_yoy
0,rainy,2012-01-01 00:00:00,173.3,NaN
1,dry,2012-02-01 00:00:00,92.3,NaN
2,rainy,2012-03-01 00:00:00,183.0,NaN
3,dry,2012-04-01 00:00:00,68.1,NaN
4,dry,2012-05-01 00:00:00,52.2,NaN
5,dry,2012-06-01 00:00:00,75.1,NaN
6,dry,2012-07-01 00:00:00,26.3,NaN
7,dry,2012-08-01 00:00:00,0.0,NaN
8,dry,2012-09-01 00:00:00,0.9,NaN
9,rainy,2012-10-01 00:00:00,170.3,NaN


## 4. The SQL SLayer generated

`--dry-run` prints the single SQL statement for the whole two-stage query without executing it — the aggregate CTE, the year-shifted self-join, the banding and regroup on top. Exactly what DuckDB ran against the remote file.

In [6]:
%%bash --out hero_sql
export SLAYER_STORAGE=.cache/cli/store
slayer query @.cache/cli/hero.json --dry-run

In [7]:
Markdown("```sql\n" + hero_sql + "\n```")

```sql
WITH monthly AS (
  SELECT
    _stage_inner."weather.date" AS "date",
    _stage_inner."weather.rain" AS "rain",
    _stage_inner."weather.rain_yoy" AS "rain_yoy"
  FROM (
    SELECT
      "weather.date",
      "weather.rain",
      "weather.rain_yoy"
    FROM (
      WITH base AS (
        SELECT
          DATE_TRUNC('MONTH', weather.date) AS "weather.date",
          CAST(SUM(weather.precipitation) AS DOUBLE) AS "weather.rain"
        FROM weather AS weather
        GROUP BY
          DATE_TRUNC('MONTH', weather.date)
      ), shifted__time_shift_inner AS (
        SELECT
          DATE_TRUNC('MONTH', weather.date) + INTERVAL '1' YEAR AS "weather.date",
          CAST(SUM(weather.precipitation) AS DOUBLE) AS "weather.rain"
        FROM weather AS weather
        GROUP BY
          DATE_TRUNC('MONTH', weather.date) + INTERVAL '1' YEAR
      ), sjoin__time_shift_inner AS (
        SELECT
          base."weather.date",
          base."weather.rain",
          shifted__time_shift_inner."weather.rain" AS "weather._time_shift_inner"
        FROM base
        LEFT JOIN shifted__time_shift_inner
          ON base."weather.date" IS NOT DISTINCT FROM shifted__time_shift_inner."weather.date"
      ), step1 AS (
        SELECT
          "weather.date",
          "weather.rain",
          "weather._time_shift_inner",
          "weather.rain" - "weather._time_shift_inner" AS "weather.rain_yoy"
        FROM sjoin__time_shift_inner
      )
      SELECT
        "weather.date",
        "weather.rain",
        "weather._time_shift_inner",
        "weather.rain_yoy"
      FROM step1
    ) AS _outer
  ) AS _stage_inner
)
SELECT
  CASE WHEN monthly.rain > 100 THEN 'rainy' ELSE 'dry' END AS "monthly.month_type",
  monthly.date AS "monthly.date",
  SUM(monthly.rain) AS "monthly.total_rain",
  SUM(monthly.rain_yoy) AS "monthly.total_rain_yoy"
FROM monthly AS monthly
GROUP BY
  CASE WHEN monthly.rain > 100 THEN 'rainy' ELSE 'dry' END,
  monthly.date
ORDER BY
  "monthly.date" ASC

```

---

The whole semantic layer over a file on the internet — a DuckDB view, auto-ingestion, and JSON queries, all from the shell. See the [Python notebook](duckdb_python_nb.ipynb) for the in-process library version, [multi-stage queries](../06_multistage_queries/multistage_queries.md) for the queries-as-models idea, and [formulas](../../concepts/formulas.md) for the full transform vocabulary.